# 03. Matrix Factorization 베이스라인

추천 시스템의 가장 기본적인 딥러닝 모델. 먼저 실행해보고 개념을 역설계합니다.

## 핵심 아이디어
```
유저-아이템 상호작용 행렬 R ≈ U · Vᵀ

R (n_users × n_items)   →   U (n_users × k) × Vᵀ (k × n_items)
            ↓
관측된 리뷰(1)와 관측 안 된 항목을 저차원 k개의 잠재 요인으로 분해
```
**잠재 요인(Latent Factor)**: 유저 취향·아이템 속성을 명시하지 않고 데이터에서 자동 학습하는 숨겨진 특성 벡터.

In [ ]:
import sys
sys.path.append('..')

import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from pathlib import Path

from models.mf import MatrixFactorization, BPRDataset
from models.evaluate import evaluate

DATA_DIR = Path('../data/processed/All_Beauty')

train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
test_df  = pd.read_parquet(DATA_DIR / 'test.parquet')
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)
with open(DATA_DIR / 'item2id.json') as f:
    item2id = json.load(f)
id2item = {v: k for k, v in item2id.items()}

meta_df = pd.read_parquet(DATA_DIR / 'meta.parquet')[['parent_asin', 'title', 'price']].drop_duplicates('parent_asin')

n_users = meta['n_users']
n_items = meta['n_items']
print(f"users: {n_users}, items: {n_items}")
print(f"train: {meta['n_train']:,}, test: {meta['n_test']:,}")

## 1. 왜 BPR Loss인가?

리뷰 데이터는 **implicit feedback** — "이 상품을 구매/리뷰했다"는 기록만 있고, "안 좋아한다"는 기록은 없습니다.

| 방식 | 문제점 |
|------|--------|
| **MSE** | 미관측 (user, item)을 전부 0으로 가정 → 단순히 리뷰 안 한 것일 뿐인데 부정으로 학습 |
| **BCE** | 위와 동일, 불균형 심각 (positive 12K vs negative 수백만) |
| **BPR** | "관측된 아이템이 관측되지 않은 아이템보다 선호된다"는 가정만 사용 → implicit에 적합 |

```
BPR Loss = -log σ(score(u, i⁺) - score(u, i⁻))
           긍정 점수가 부정 점수보다 높아질수록 loss → 0
```

In [ ]:
# BPR Loss 시각화
diff = torch.linspace(-3, 3, 100)
bpr_loss = -F.logsigmoid(diff)

plt.figure(figsize=(6, 3))
plt.plot(diff.numpy(), bpr_loss.numpy(), color='steelblue')
plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='score_pos = score_neg')
plt.xlabel('score_pos − score_neg')
plt.ylabel('BPR Loss')
plt.title('BPR Loss: 긍정-부정 점수 차이가 클수록 loss 감소')
plt.legend()
plt.tight_layout()
plt.show()

## 2. 모델 구조 확인

In [ ]:
n_factors = 64
model = MatrixFactorization(n_users, n_items, n_factors)
print(model)
print(f"\n파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"  user_emb : {n_users} × {n_factors} = {n_users * n_factors:,}")
print(f"  item_emb : {n_items} × {n_factors} = {n_items * n_factors:,}")
print()
print("예측 방식: user_emb[u] · item_emb[i]  (내적 = 유사도 점수)")

## 3. 학습

In [ ]:
dataset   = BPRDataset(train_df, n_items)
loader    = DataLoader(dataset, batch_size=512, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []
for epoch in range(1, 51):
    model.train()
    total_loss = sum(
        model.bpr_loss(u, pi, ni).item()
        for u, pi, ni in (
            (optimizer.zero_grad() or b)
            for b in (
                (lambda b: (model.bpr_loss(*b).backward() or optimizer.step() or b))(batch)
                for batch in loader
            )
        )
    ) / len(loader)
    history.append({'epoch': epoch, 'loss': total_loss})
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss={total_loss:.4f}")

In [ ]:
# 위 셀이 복잡하면 아래 단순 버전 사용
model = MatrixFactorization(n_users, n_items, n_factors)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(1, 51):
    model.train()
    total_loss = 0.0
    for users, pos_items, neg_items in loader:
        optimizer.zero_grad()
        loss = model.bpr_loss(users, pos_items, neg_items)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    history.append({'epoch': epoch, 'loss': avg_loss})
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss={avg_loss:.4f}")

In [ ]:
hist_df = pd.DataFrame(history)
plt.figure(figsize=(7, 3))
plt.plot(hist_df['epoch'], hist_df['loss'], color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('BPR Loss')
plt.title('Training Loss')
plt.tight_layout()
plt.show()

## 4. 평가: Hit@10, NDCG@10

### 평가 프로토콜 (Leave-one-out + 99 negatives)
```
각 유저에 대해:
  1. test 아이템 1개 (실제 마지막 상호작용)
  2. 랜덤 부정 아이템 99개 (유저가 본 적 없는 것)
  → 100개 후보 중 test 아이템의 순위를 매김

Hit@10  : 상위 10위 안에 있으면 1, 아니면 0
NDCG@10 : 순위에 따른 가중치 (1위=1.0, 2위=0.63, 10위=0.29)
```

In [ ]:
model.eval()

def score_fn(user_id, item_ids):
    u = torch.tensor([user_id] * len(item_ids))
    i = torch.tensor(item_ids)
    with torch.no_grad():
        return model(u, i)

metrics = evaluate(score_fn, train_df, test_df, n_items, K=10)
print("=== MF 평가 결과 ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# 랜덤 베이스라인과 비교
def random_score_fn(user_id, item_ids):
    return torch.rand(len(item_ids))

random_metrics = evaluate(random_score_fn, train_df, test_df, n_items, K=10)
print("\n=== 랜덤 베이스라인 ===")
for k, v in random_metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\n개선율: Hit@10 {metrics['hit@10']/random_metrics['hit@10']:.1f}×")

## 5. 임베딩 시각화 (PCA)

학습된 아이템 임베딩을 2D로 축소해 유사한 아이템이 가까이 모이는지 확인합니다.  
잘 학습된 MF라면 비슷한 성격의 아이템끼리 클러스터를 이룹니다.

In [ ]:
item_embs = model.item_emb.weight.detach().numpy()   # (n_items, 64)
pca = PCA(n_components=2)
coords = pca.fit_transform(item_embs)                # (n_items, 2)

# 리뷰 수 많은 상위 아이템만 레이블 표시
popular_items = train_df['parent_asin'].value_counts().head(20).index.tolist()

plt.figure(figsize=(10, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=10, color='steelblue')

for item_id in popular_items:
    asin = id2item.get(item_id, '')
    title = meta_df[meta_df['parent_asin'] == asin]['title'].values
    label = str(title[0])[:20] if len(title) > 0 else str(asin)
    plt.annotate(label, coords[item_id], fontsize=7, alpha=0.8)
    plt.scatter(coords[item_id, 0], coords[item_id, 1], s=40, color='coral', zorder=5)

plt.title(f'Item Embeddings PCA (설명 분산: {pca.explained_variance_ratio_.sum():.2%})')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

## 6. 샘플 추천 결과 확인

In [ ]:
def recommend(user_id: int, top_k: int = 5) -> pd.DataFrame:
    """유저의 학습 이력 제외 후 top-K 추천."""
    seen = set(train_df[train_df['user_id'] == user_id]['parent_asin'].tolist())
    scores = model.score_all_items(user_id).numpy()

    item_scores = [
        (item_id, scores[item_id])
        for item_id in range(n_items)
        if item_id not in seen
    ]
    item_scores.sort(key=lambda x: -x[1])

    rows = []
    for rank, (item_id, score) in enumerate(item_scores[:top_k], 1):
        asin = id2item.get(item_id, '')
        row = meta_df[meta_df['parent_asin'] == asin]
        title = row['title'].values[0] if len(row) > 0 else 'N/A'
        price = row['price'].values[0] if len(row) > 0 else 'N/A'
        rows.append({'rank': rank, 'score': round(float(score), 3), 'title': str(title)[:50], 'price': price})
    return pd.DataFrame(rows)

# 유저 0의 학습 이력
user_id = 0
history_items = train_df[train_df['user_id'] == user_id].sort_values('timestamp')
history_asins = [id2item[i] for i in history_items['parent_asin']]
history_titles = [meta_df[meta_df['parent_asin'] == a]['title'].values for a in history_asins]
print(f"유저 {user_id}의 학습 이력:")
for i, t in enumerate(history_titles):
    print(f"  {i+1}. {str(t[0])[:50] if len(t) > 0 else '?'}")

print(f"\nMF 추천 결과 (top-5):")
display(recommend(user_id, top_k=5))

---
## 요약 — MF가 학습하는 것과 한계

### MF가 학습하는 것
- 유저의 **전체 이력 평균적 취향** → 유저 임베딩
- 아이템의 **평균적 특성** → 아이템 임베딩
- 두 벡터의 내적으로 친밀도(선호도) 예측

### MF의 한계
| 한계 | 설명 |
|------|------|
| **선형성** | 내적은 선형 연산 → 복잡한 유저-아이템 상호작용 포착 어려움 |
| **정적 선호** | 취향 변화를 반영 못함 (최근 관심사 vs. 오래된 관심사 구분 없음) |
| **Cold-start** | 새 유저/아이템은 임베딩이 없어 추천 불가 |

→ **다음 스텝**: NCF — MLP로 비선형 상호작용을 추가해 MF의 선형성 한계를 극복